![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 1 · Tarea 1 · 20 % · entrega hasta el vie 25/09, 23:59</div><div style="font-size:22px;font-weight:700;margin-top:4px">T1 · Flujo end-to-end reproducible para un caso de TI</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Qué entregas** | **Este mismo notebook**, ejecutado de principio a fin y guardado con sus salidas, renombrado `T1_Apellido_Nombre.ipynb`. No hay informe en PDF. |
| **Caso** | Un proveedor de telecomunicaciones quiere anticipar qué clientes cancelarán su contrato para ofrecerles una acción de retención. |
| **Datos** | Telco Customer Churn (OpenML `data_id=42178`): 7043 clientes, 19 variables de tipos mixtos, ≈ 26 % de abandono. |
| **Costos del caso** | Contactar a un cliente que no se iba (FP) cuesta ≈ USD 20; perder a un cliente (FN) cuesta ≈ USD 100. Relación 5:1 (supuestos didácticos; puedes justificar otros). |
| **Resultado de aprendizaje** | RDA1 · competencias CG-G1 y CE-G1 |
| **Puedes reutilizar** | E1.1 (proceso y baseline) · E1.2 (métricas y umbral por costo) · E1.3 (pipeline y fuga de datos) |

## Cómo se califica

Cada sección de este notebook es un criterio de la rúbrica y lleva su puntaje en el título. En cada una hay:

- **Qué hacer**: los pasos exactos que se esperan.
- Celdas de código con `# TODO`.
- Una celda **Tu análisis**: ahí escribes la interpretación. *El código que corre sin análisis no suma puntos.*
- A veces una celda de **autoverificación** con `assert`: si pasa, vas bien.

**Uso de IA.** Puedes usar IA agéntica (Claude Code, Codex, Gemini CLI) o de chat (ChatGPT, Claude, Gemini) para
resolver esta tarea, **siempre que lo declares en la sección 8**, indicando en qué secciones la usaste. Este curso
no es de programación en Python: se evalúan tus decisiones, su justificación y cómo verificaste los resultados.

**Antes de entregar**: menú *Kernel → Restart & Run All* (o *Entorno de ejecución → Reiniciar y ejecutar todo*),
revisa que todas las celdas tengan salida y guarda el archivo con tu nombre.

## 0 · Configuración y datos (no puntúa, pero es obligatorio)

**Qué hacer:** ejecuta esta celda tal cual. Fija la semilla, deja constancia de las versiones y carga los datos por
código (nunca subas un CSV modificado a mano).

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler
from sklearn.datasets import fetch_openml

SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"Python {sys.version.split()[0]} · scikit-learn {sklearn.__version__} · pandas {pd.__version__} · semilla {SEED}")

datos = fetch_openml(data_id=42178, as_frame=True).frame
y = (datos.pop("Churn") == "Yes").astype(int)
X = datos.drop(columns=["customerID"], errors="ignore")
X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")   # los blancos pasan a NaN
print(f"{len(X)} clientes · {X.shape[1]} variables · {y.mean():.1%} de abandono")
X.head(3)

## 1 · Calidad de datos y partición sin fuga · 20 puntos

**Qué hacer:**

1. Reporta tipos de variable, faltantes por columna, duplicados, cardinalidad de las categóricas y balance de clases.
2. Explica qué haces con los faltantes de `TotalCharges` y por qué (¿son un error o significan algo?).
3. **Separa la prueba (20 %, estratificada) antes de cualquier preprocesamiento** y no vuelvas a tocarla hasta la
   sección 5.

*Reutiliza:* el reporte de calidad de E1.3 y la partición de E1.1.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO: reporte de calidad. Al menos: X.dtypes, X.isna().sum(), X.duplicated().sum(),
# la cardinalidad de las columnas de tipo object y y.value_counts(normalize=True).


# TODO: separa la prueba estratificada (20 %, random_state=SEED).
# X_train, X_test, y_train, y_test = ...

In [ ]:
# Autoverificación de la sección 1
assert "X_train" in dir() and "X_test" in dir(), "Falta la partición."
assert len(X_train) + len(X_test) == len(X), "La partición no cubre todas las filas."
assert abs(y_train.mean() - y_test.mean()) < 0.02, "La partición no parece estratificada."
print("✓ Partición correcta:", len(X_train), "entrenamiento ·", len(X_test), "prueba")

**Tu análisis (sección 1).** ¿Qué problemas de calidad encontraste? ¿Qué decidiste hacer con los faltantes de
`TotalCharges` y por qué? ¿Por qué la prueba se separa antes de preprocesar?

*(escribe aquí)*

## 2 · Pipeline y ColumnTransformer · 15 puntos

**Qué hacer:**

1. Arma un `ColumnTransformer` con una rama numérica (imputación + escalado) y otra categórica
   (imputación + `OneHotEncoder(handle_unknown="ignore")`).
2. Envuélvelo en un `Pipeline` junto con el modelo. **Nada de preprocesamiento fuera del pipeline.**

*Reutiliza:* el pipeline de E1.3 · Nivel 3.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# TODO: listas de columnas numéricas y categóricas (a partir de X_train.dtypes).
# num = ...
# cat = ...

# TODO: preprocesamiento (ColumnTransformer) y un pipeline con LogisticRegression(max_iter=1000).
# prep = ...
# logistica = ...

In [ ]:
# Autoverificación de la sección 2
assert isinstance(logistica, Pipeline), "El modelo debe ser un Pipeline."
assert any(isinstance(p, ColumnTransformer) for _, p in logistica.steps), "Falta el ColumnTransformer."
print("✓ Pipeline listo:", [n for n, _ in logistica.steps])

**Tu análisis (sección 2).** ¿Qué hace cada rama del `ColumnTransformer` y por qué elegiste esa imputación?
¿Qué pasaría si escalaras antes de partir los datos?

*(escribe aquí)*

## 3 · Modelo de referencia, dos modelos y validación cruzada · 15 puntos

**Qué hacer:**

1. Reporta un `DummyClassifier` como modelo de referencia.
2. Entrena **dos modelos**: la regresión logística de la sección 2 y otro de tu elección (árbol, bosque, boosting…).
3. Evalúa los tres con **validación cruzada estratificada de 5 folds** sobre el entrenamiento y reporta
   **media ± desviación** de al menos dos métricas (por ejemplo, AUC y F1 o PR-AUC).

*Reutiliza:* la comparación con baseline de E1.1 · Nivel 3 y la validación cruzada de E1.2.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# TODO: define el baseline y tu segundo modelo (también dentro de un Pipeline).
# baseline = ...
# modelo2 = ...

# TODO: evalúa los tres con cross_validate y arma una tabla con media ± desviación.
# tabla_cv = ...

**Tu análisis (sección 3).** ¿Cuánto mejora cada modelo al modelo de referencia? ¿La diferencia entre tus dos
modelos es mayor que la variabilidad entre folds? ¿Con cuál te quedas **por ahora** y por qué?

*(escribe aquí)*

## 4 · Métrica y umbral por costo · 15 puntos

**Qué hacer:**

1. Escribe la matriz de costos del caso (FP ≈ USD 20, FN ≈ USD 100, o los que justifiques).
2. Obtén **probabilidades fuera de muestra** del modelo elegido (`cross_val_predict(..., method="predict_proba")`).
3. Barre umbrales de 0.05 a 0.95, calcula el **costo esperado por cliente** y elige el umbral que lo minimiza.
   **Sin usar la prueba.**
4. Grafica el costo frente al umbral y marca el elegido.

*Reutiliza:* el barrido de umbrales de E1.2 · Nivel 3.

In [ ]:
from sklearn.model_selection import cross_val_predict

COSTO_FP, COSTO_FN = 20, 100        # TODO: ajusta y justifica si usas otros

# TODO: probabilidades fuera de muestra, barrido de umbrales, costo esperado y elección del umbral.
# p_oof = ...
# UMBRAL = ...

# TODO: gráfico del costo esperado frente al umbral, con una línea en el umbral elegido.

In [ ]:
# Autoverificación de la sección 4
assert 0 < UMBRAL < 1, "El umbral debe estar entre 0 y 1."
print(f"✓ Umbral elegido: {UMBRAL:.2f} (umbral teórico con probabilidades calibradas: {COSTO_FP / (COSTO_FP + COSTO_FN):.2f})")

**Tu análisis (sección 4).** ¿Qué métrica es la adecuada en este caso y por qué? ¿Por qué el umbral no es 0.5?
¿Cuánto cambia el costo entre el umbral 0.5 y el tuyo?

*(escribe aquí)*

## 5 · Evaluación final en la prueba · 5 puntos

**Qué hacer:** entrena tu modelo elegido con **todo** el entrenamiento y evalúalo **una sola vez** en la prueba, con
el umbral de la sección 4. Reporta matriz de confusión, precisión, recall, F1, AUC y el costo por cliente.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# TODO: entrena con todo el entrenamiento, predice en la prueba con tu umbral y reporta las métricas y el costo.

**Tu análisis (sección 5).** ¿Los resultados en la prueba se parecen a los de la validación cruzada? Si no,
¿qué lo explica?

*(escribe aquí)*

## 6 · Errores por segmento y limitaciones · 10 puntos

**Qué hacer:**

1. Calcula el desempeño (recall y precisión, o la tasa de error) por **tipo de contrato**, **antigüedad**
   (`tenure` en tramos) y **método de pago**.
2. Identifica dónde falla más el modelo y plantea una hipótesis de por qué.
3. Enumera al menos tres limitaciones o riesgos de usar este modelo para decidir a quién contactar.

In [ ]:
# TODO: tabla de desempeño por segmento (usa X_test y las predicciones de la sección 5).

**Tu análisis (sección 6).** ¿En qué segmento falla más? ¿Por qué crees que pasa? ¿Qué riesgos tiene usar este
modelo en la práctica (clientes tratados de forma distinta, datos que cambian, costos mal estimados)?

*(escribe aquí)*

## 7 · Conclusión y recomendación · 10 puntos

**Qué hacer:** en 150–250 palabras, responde como si escribieras al equipo de retención: qué modelo recomiendas,
con qué umbral, cuánto mejora frente a no hacer nada (el modelo de referencia), qué cuesta equivocarse y qué
harías antes de ponerlo en producción.

**Tu recomendación.**

*(escribe aquí)*

## 8 · Declaración de uso de IA (obligatoria)

Completa la tabla. Si no usaste IA, escribe "No usé IA" y firma igual. Omitirla o declararla de forma falsa sí
afecta la calificación (norma f del sílabo).

| | |
|---|---|
| **Herramientas** | *(por ejemplo: ChatGPT, Claude Code; o "ninguna")* |
| **Secciones donde la usé** | *(por ejemplo: sección 2 para el ColumnTransformer y sección 6 para el gráfico)* |
| **Para qué** | *(escribir código, depurar un error, redactar el análisis, revisar mi interpretación…)* |
| **Prompts relevantes** | *(pega los 2 o 3 más importantes)* |
| **Qué verifiqué yo** | *(ejecuté todo de cero, comprobé que no hay fuga, revisé las cifras contra las salidas…)* |
| **Qué corregí o descarté** | *(qué te propuso la IA que no usaste y por qué)* |

**Autoría.** El análisis, las decisiones y las conclusiones de este notebook son míos.

Nombre: *(tu nombre)* · Fecha: *(fecha de entrega)*

## Lista de cotejo antes de entregar

- [ ] Reinicié el kernel y ejecuté todo de arriba hacia abajo, sin errores.
- [ ] Todas las celdas de código tienen su salida visible.
- [ ] Respondí **todas** las celdas "Tu análisis" (no basta con el código).
- [ ] La prueba se usa una sola vez, en la sección 5.
- [ ] El umbral está justificado con los costos, no es 0.5 por defecto.
- [ ] Completé la declaración de uso de IA.
- [ ] Guardé el archivo como `T1_Apellido_Nombre.ipynb` y lo subí al LMS.

In [ ]:
from datetime import datetime

print(f"Notebook ejecutado el {datetime.now():%Y-%m-%d %H:%M} · scikit-learn {sklearn.__version__} · semilla {SEED}")